# 🛡️ DeepShield Audio — Notebook 03: Model Training

This notebook demonstrates how to train the models using the `tf.data` pipeline and the trainer script.

> **Note:** For full training on the 10GB ASVspoof dataset, it is highly recommended to run the training script from the command line using a GPU: `python -m src.trainer --model custom_cnn`

In [1]:
import sys
sys.path.insert(0, '..')

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import os

from src.models.custom_cnn import build_custom_cnn
from src.models.cnn_lstm import build_cnn_lstm
from src.models.efficientnet import build_efficientnet_b0
from src.trainer import get_callbacks
from src.dataset import build_tf_dataset
from src.data_parser import make_synthetic_dataframe

print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

ImportError: cannot import name 'get_callbacks' from 'src.trainer' (/Users/yash/Documents/Projects/Audio-Deepfake-Detection/notebooks/../src/trainer.py)

## 1. Prepare Synthetic Data (for demonstration)
Since we might not have the full dataset available in this environment, we'll generate synthetic data to demonstrate the training loop.

In [ ]:
# Generate synthetic DataFrame
train_df = make_synthetic_dataframe(n_bonafide=50, n_spoof=50)
val_df   = make_synthetic_dataframe(n_bonafide=20, n_spoof=20)

print(f'Train samples: {len(train_df)}')
print(f'Val samples: {len(val_df)}')

# Create tf.data.Dataset (this will extract log-mel on the fly!)
# Note: On synthetic data where files don't exist, this will error out.
# We will use dummy tensors for the actual `model.fit` demonstration.
print('Dataset pipelines can be built like this:')
print('train_ds = build_tf_dataset(train_df, batch_size=32, is_training=True)')
print('val_ds = build_tf_dataset(val_df, batch_size=32, is_training=False)')

## 2. Model Summaries
Let's look at the architectures we've implemented.

In [ ]:
print('--- CUSTOM CNN ---')
cnn_model = build_custom_cnn()
cnn_model.summary()

print('\n\n--- CNN-LSTM ---')
lstm_model = build_cnn_lstm()
lstm_model.summary()

## 3. Dummy Training Loop Demonstration
We create some random tensors to simulate a batch of data and run `model.fit()` for 2 epochs.

In [ ]:
import numpy as np
from src.config import N_MELS, T_FRAMES

# Generate random noise simulating Log-Mel spectrograms
X_dummy = np.random.randn(64, N_MELS, T_FRAMES, 1).astype(np.float32)
y_dummy = np.random.randint(0, 2, size=(64, 1)).astype(np.float32)

callbacks = get_callbacks(model_name='demo_cnn')

history = cnn_model.fit(
    X_dummy, y_dummy,
    batch_size=16,
    epochs=5,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

## 4. Plot Learning Curves

In [ ]:
def plot_learning_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    fig.patch.set_facecolor('#1E1E2E')
    
    for ax in [ax1, ax2]:
        ax.set_facecolor('#1E1E2E')
        ax.tick_params(colors='white')
        ax.xaxis.label.set_color('white')
        ax.yaxis.label.set_color('white')
        ax.title.set_color('white')
        
    # Loss
    ax1.plot(history.history['loss'], label='Train Loss', color='#4ECDC4')
    ax1.plot(history.history['val_loss'], label='Val Loss', color='#E63946')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend(facecolor='#2A2A35', labelcolor='white')
    
    # Accuracy
    ax2.plot(history.history['accuracy'], label='Train Acc', color='#4ECDC4')
    ax2.plot(history.history['val_accuracy'], label='Val Acc', color='#E63946')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend(facecolor='#2A2A35', labelcolor='white')
    
    plt.show()

plot_learning_curves(history)